In [111]:
import requests
import json
from datetime import datetime
import time
import random
import asyncio, aiohttp



In [115]:

API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"

async def fetch_reposters(session, uri, limit=100):
    """Fetch reposters for one post URI safely."""
    params = {"uri": uri, "limit": limit}
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        async with session.get(API_URL, params=params, headers=headers, timeout=100) as r:
            if r.status != 200:
                print(f"⚠️ HTTP {r.status} for {uri}")
                return []
            data = await r.json()
            return [u["did"] for u in data.get("repostedBy", [])]
    except Exception as e:
        print(f"Error fetching {uri}: {e}")
        return []


async def collect_all_reposters(file_path, concurrency=25):
    """Fast async fetch of reposters for posts with repostCount > 0."""
    posts, tasks = [], []

    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    async with aiohttp.ClientSession(connector=connector) as session:
        # Load posts and prepare tasks
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    post = json.loads(line)
                except json.JSONDecodeError:
                    continue

                # Find repost count — supports slightly different schemas
                repost_count = (
                    post.get("repostCount")
                    or post.get("metrics", {}).get("repostCount")
                    or post.get("post", {}).get("repostCount")
                    or 0
                )

                if repost_count > 0:
                    uri = post.get("uri") or post.get("post", {}).get("uri")
                    if uri:
                        posts.append(post)
                        tasks.append(fetch_reposters(session, uri))

        # Run all requests concurrently (only for posts that were reposted)
        repost_lists = await asyncio.gather(*tasks, return_exceptions=True)

    # Merge results
    results = {}
    for i, (post, reposters) in enumerate(zip(posts, repost_lists)):
        if not reposters:
            continue
        post_id = (
            post.get("uri")
            or post.get("post", {}).get("uri")
            or f"post_{i}"
        )
        results[post_id] = {"post": post, "reposters": reposters}

    return results


In [116]:
repost_dict = await collect_all_reposters("breachdallas.jsonl")


In [118]:
print(len(repost_dict))

578


In [133]:


REL_URL = "https://public.api.bsky.app/xrpc/app.bsky.graph.getRelationships"

async def check_follow(session, reposter, author):
    """Return True/False if reposter follows author, None on error."""
    params = {"actor": reposter, "others": author}
    try:
        async with session.get(REL_URL, params=params, timeout=100) as r:
            if r.status != 200:
                print("Error")
                return None
            data = await r.json()
            rels = data.get("relationships", [])
            if not rels:
                print("Error")
                return None
            follows = rels[0].get("following", False)
            # Handle both bools and at:// strings
            if isinstance(follows, str) and follows.startswith("at://"):
                return True
            return bool(follows)
    except Exception:
        print("Error")
        return None


async def check_all_follows(repost_results, concurrency=25):
    """Return {author_did: {reposter_did: True/False}} for all posts."""
    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    follows_map = {}

    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = []
        mapping = []  # [(author, reposter)] in same order as tasks

        # Create all tasks
        for post_data in repost_results.values():
            author = (
                post_data["post"].get("author", {})
                or post_data["post"].get("post", {}).get("author", {})
            ).get("did")
            if not author:
                continue
            for reposter in post_data.get("reposters", []):
                mapping.append((author, reposter))
                tasks.append(asyncio.create_task(check_follow(session, reposter, author)))

        print(f"🔍 Checking {len(tasks)} reposter→author relationships...")

        # Run concurrently
        results = await asyncio.gather(*tasks, return_exceptions=True)

        # Combine results

        for (author, rep), follows in zip(mapping, results):
            if isinstance(follows, bool):  # only keep successful ones

                follows_map.setdefault(author, {})[rep] = follows

    print(f"✅ Done. Collected {sum(len(v) for v in follows_map.values())} valid relationships "
          f"for {len(follows_map)} authors total.")
    return follows_map


In [132]:
followers = await check_all_follows(repost_dict)

🔍 Checking 1044 reposter→author relationships...
1044
✅ Done. Collected 383 valid relationships for 90 authors total.


In [137]:
print((followers))

{'did:plc:3jod2mt7f7jva63p5wagydbv': {'did:plc:oouxpk5mwn55tcyko2flvh7d': False}, 'did:plc:2f5xrk2cvjsds5vu46eik3as': {'did:plc:gntc2im4bglfkgaiktcrem4y': False}, 'did:plc:xhc2onkmmhnzawgujmyr5mx4': {'did:plc:eqvvek25ndbzzz67mkreuqrt': False}, 'did:plc:b2cbyqmmyenguxzawppoyflr': {'did:plc:wjswtzcbfczkp4jqsauedd5d': False}, 'did:plc:tat3viqslfd4ffydmedq5cpp': {'did:plc:vwop6am56fyupc5idkstdic2': True, 'did:plc:pz4mhojh6iaa62bfvrkrirdp': True, 'did:plc:m5vod53e3f7atminarpifjdh': True}, 'did:plc:pgpeu5uqi4so6kxvbzpuxfg2': {'did:plc:pltcigp6ksv4k4nzprn3xa5q': False, 'did:plc:eqvvek25ndbzzz67mkreuqrt': False}, 'did:plc:q5ftblgwbdlgm7dy6iuw6d6j': {'did:plc:rnqmgvpbmk5pyfrnngpkrj2c': False, 'did:plc:y626yjzy56afehdxahpromq5': False}, 'did:plc:33ykcmkbsplmo3khippdy32n': {'did:plc:ghv7vwxuvz6ekw7yxdjqkxck': True}, 'did:plc:mfcu72jhgelckt2ln4xaxbrt': {'did:plc:eb7ubjuks3br2bacmumtmf4g': True, 'did:plc:uf4gqrodw2w4c5dw2fwxjqx4': True, 'did:plc:rnqmgvpbmk5pyfrnngpkrj2c': False, 'did:plc:dh4krmqw7p

In [134]:
def compute_follow_ratio(follows_map):
    """Compute the overall ratio of True / (True + False) across all authors."""
    total_true = 0
    total_count = 0

    for reposters in follows_map.values():
        for follows in reposters.values():
            total_count += 1
            if follows:
                total_true += 1

    return total_true / total_count if total_count > 0 else 0.0


compute_follow_ratio(followers)

0.618798955613577

In [138]:
from collections import defaultdict, OrderedDict

def analyze_false_follow_ratios(follows_map):
    """
    Compute average % of False relationships (non-followers) among reposts,
    grouped by number of reposters per author, sorted by reposter count.

    Args:
        follows_map (dict): {author_did: {reposter_did: True/False}}

    Returns:
        dict: Ordered {num_reposters: avg_false_percentage}, sorted ascending.
    """
    grouped_ratios = defaultdict(list)

    for author, rels in follows_map.items():
        if not rels:
            continue

        true_count = sum(1 for f in rels.values() if f is True)
        false_count = sum(1 for f in rels.values() if f is False)
        total = true_count + false_count

        if total == 0:
            continue

        false_pct = (false_count / total) * 100
        grouped_ratios[len(rels)].append(false_pct)

    # Compute averages and sort by reposter count
    avg_by_group = {
        count: sum(vals) / len(vals)
        for count, vals in grouped_ratios.items()
    }

    # Return sorted dictionary
    return dict(sorted(avg_by_group.items(), key=lambda x: x[0]))
